# Heart Disease Prediction
## ML Deployment Project

---

## 1. Problem Definition

### ปัญหาคืออะไร?
โรคหัวใจ (Cardiovascular Disease) เป็นสาเหตุการเสียชีวิตอันดับ 1 ของโลก คร่าชีวิตประมาณ 17.9 ล้านคนต่อปี (WHO) การตรวจพบความเสี่ยงตั้งแต่เนิ่นๆ สามารถช่วยให้ผู้ป่วยได้รับการรักษาทันเวลาและลดอัตราการเสียชีวิตได้

### ทำไมถึงน่าแก้ไข?
- การวินิจฉัยโรคหัวใจในปัจจุบันต้องอาศัยการตรวจหลายขั้นตอนที่มีค่าใช้จ่ายสูง
- ML model สามารถช่วย **คัดกรองเบื้องต้น** จากข้อมูลพื้นฐาน เช่น อายุ ความดันโลหิต คอเลสเตอรอล ผล ECG ได้
- ช่วยให้แพทย์มีเครื่องมือสนับสนุนการตัดสินใจ (Decision Support Tool) ไม่ใช่ทดแทนแพทย์

### ทำไม ML ถึงเหมาะกับปัญหานี้?
- เป็น **Binary Classification** ที่ชัดเจน (เป็นโรคหัวใจ / ไม่เป็น)
- มี features ที่เป็นตัวแปรทางการแพทย์ที่มีความสัมพันธ์กับโรคหัวใจตามหลักวิชาการ
- Dataset มีขนาดเพียงพอ (918 ตัวอย่าง) สำหรับ traditional ML
- ความสัมพันธ์ระหว่าง features กับ target อาจเป็น non-linear ซึ่ง ML จัดการได้ดีกว่า rule-based

## 2. Dataset Description

### ที่มาของข้อมูล
Dataset นี้รวบรวมจาก 5 แหล่งข้อมูลโรคหัวใจที่มีชื่อเสียง:
1. Cleveland Clinic Foundation
2. Hungarian Institute of Cardiology
3. V.A. Medical Center, Long Beach
4. University Hospital, Zurich
5. Statlog (Heart) Dataset

เผยแพร่บน [Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) โดยรวม 918 ตัวอย่างหลังลบข้อมูลซ้ำ

### Features ทั้งหมด 11 ตัว + 1 Target

| Feature | Type | Description | ความหมายทางการแพทย์ |
|---------|------|-------------|---------------------|
| **Age** | Numerical | อายุผู้ป่วย (ปี) | ความเสี่ยงโรคหัวใจเพิ่มขึ้นตามอายุ |
| **Sex** | Categorical (M/F) | เพศ | ผู้ชายมีความเสี่ยงสูงกว่าในช่วงอายุเท่ากัน |
| **ChestPainType** | Categorical | ประเภทอาการเจ็บหน้าอก | ASY (ไม่มีอาการ) พบในผู้ป่วยโรคหัวใจบ่อย |
| **RestingBP** | Numerical | ความดันโลหิตขณะพัก (mm Hg) | ความดันสูง = ปัจจัยเสี่ยง |
| **Cholesterol** | Numerical | คอเลสเตอรอลในเลือด (mg/dl) | ค่าสูงเกินไป = ปัจจัยเสี่ยง |
| **FastingBS** | Binary (0/1) | น้ำตาลในเลือดขณะอดอาหาร > 120 mg/dl | เบาหวานเป็นปัจจัยเสี่ยงโรคหัวใจ |
| **RestingECG** | Categorical | ผลตรวจคลื่นไฟฟ้าหัวใจขณะพัก | Normal / ST-T wave abnormality / LVH |
| **MaxHR** | Numerical | อัตราการเต้นหัวใจสูงสุดที่ทำได้ | ค่าต่ำ = อาจบ่งบอกปัญหาหัวใจ |
| **ExerciseAngina** | Binary (Y/N) | เจ็บหน้าอกขณะออกกำลังกาย | ถ้ามี = สัญญาณเตือนสำคัญ |
| **Oldpeak** | Numerical | ST depression จากการออกกำลังกาย | ค่าสูง = หัวใจขาดเลือด |
| **ST_Slope** | Categorical | ความชันของ ST segment | Flat/Down = ผิดปกติ |
| **HeartDisease** | **Target** (0/1) | 0 = ปกติ, 1 = เป็นโรคหัวใจ | - |

### ประเภท ChestPainType
- **TA** (Typical Angina): เจ็บหน้าอกแบบ classic — เจ็บเมื่อออกแรง หายเมื่อพัก
- **ATA** (Atypical Angina): เจ็บหน้าอกแบบไม่ตรงตำรา
- **NAP** (Non-Anginal Pain): เจ็บหน้าอกที่ไม่เกี่ยวกับหัวใจ
- **ASY** (Asymptomatic): ไม่มีอาการเจ็บหน้าอก — พบบ่อยในผู้ป่วยโรคหัวใจจริงๆ

## 3. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('heart.csv')
print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# ดู unique values ของ categorical features
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for col in cat_cols:
    print(f'{col}: {df[col].unique()} (n={df[col].nunique()})')

In [ ]:
# ดู class balance ของ target
print('Target Distribution:')
print(df['HeartDisease'].value_counts())
print(f'\nRatio: {df["HeartDisease"].value_counts(normalize=True).round(3).to_dict()}')